# Default probabily 

## Parameters

In [17]:
import pandas as pd
import os
import logging
import datetime

from sklearn.model_selection import GridSearchCV, train_test_split ,RandomizedSearchCV

import json
import numpy as np
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
import lightgbm as lgb
import numpy as np

import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt

import joblib


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

# Construcción de un pipeline para los atributos numéricos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer



# ref of data set https://github.com/buddhiniK/loan-default-prediction/tree/main
# Parameters
Filename = 'source.xlsx'
Thresh_parameter = 9  # number of non-null values required to keep a row

# Choose the month and year for the analysis
year = 2026
month = 2

# When it is true we are going to calculate again the best model
flag_best_model = False

theshdropvalues = 0.05 # maximum threshold percentage of rows that can be dropped

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

# Handle to write logs to a file
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)

results_path = os.path.join('Results', f'{year}', f'{month}')
os.makedirs(results_path, exist_ok=True)

date_param = datetime.date(int(year), int(month), 1)
    

date_param  = pd.to_datetime(date_param)


date_param = date_param + pd.offsets.MonthEnd(0)

# ============================================
#  DICTIONARY of models 
# ============================================

dict_model = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    
    'Naive Bayes': GaussianNB(),
    
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
    
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    
    'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=42),
    
    'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42),
    
    'SVM (calibrado)': CalibratedClassifierCV(SVC(kernel='rbf', probability=False), cv=5),  
    # Distance to margin (not probability)
    
    'KNN (calibrado)': CalibratedClassifierCV(KNeighborsClassifier(n_neighbors=15), cv=5),
    # Proportion of neighbors (discrete, no distance weighting)
}

# ============================================
# GRIDS of hyperparameters to optimize for each model
# ============================================

# we have commented some parameters to reduce the number of combinations and speed up the process, but in real life we would consider them all for a more thorough optimization.

param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10]
        # ,
        # 'penalty': ['l2'],
        # 'solver': ['lbfgs', 'liblinear']
    },
    
    'Naive Bayes': {
        'var_smoothing': [1e-9, 1e-8, 1e-7]
    },
    
    'Decision Tree': {
        'max_depth': [5, 7, 10, 15]
        # ,
        # 'min_samples_split': [10, 20, 50],
        # 'min_samples_leaf': [5, 10, 20],
        # 'criterion': ['gini', 'entropy']
    },
    
    'Random Forest': {
        'n_estimators': [100, 200, 300]
        # ,
        # 'max_depth': [10, 15, 20],
        # 'min_samples_split': [5, 10, 20],
        # 'min_samples_leaf': [2, 5, 10],
        # 'max_features': ['sqrt', 'log2']
    },
    
    'Gradient Boosting': {
        'n_estimators': [100, 150, 200]
        # ,
        # 'learning_rate': [0.05, 0.1, 0.15],
        # 'max_depth': [3, 4, 5],
        # 'min_samples_split': [20, 50],
        # 'min_samples_leaf': [10, 20],
        # 'subsample': [0.8, 1.0]
    },
    
    'XGBoost': {
        'n_estimators': [100, 150, 200]
        # ,
        # 'max_depth': [3, 5, 7],
        # 'learning_rate': [0.05, 0.1, 0.15],
        # 'subsample': [0.8, 1.0],
        # 'colsample_bytree': [0.8, 1.0],
        # 'gamma': [0, 0.1, 0.2]
    },
    
    'LightGBM': {
        'n_estimators': [100, 150, 200]
        # ,
        # 'max_depth': [3, 5, 7, -1],
        # 'learning_rate': [0.05, 0.1, 0.15],
        # 'num_leaves': [15, 31, 63],
        # 'subsample': [0.8, 1.0],
        # 'colsample_bytree': [0.8, 1.0]
    },
    
    'SVM (calibrado)': {
        'estimator__C': [0.1, 1, 10]
        # ,
        # 'base_estimator__kernel': ['rbf'],
        # 'base_estimator__gamma': ['scale', 0.1, 1],
        # 'cv': [3],
        # 'method': ['sigmoid']
    },
    
    'KNN (calibrado)': {
        'base_estimator__n_neighbors': [5, 11, 15, 21]
        # ,
        # 'base_estimator__weights': ['uniform', 'distance'],
        # 'base_estimator__p': [2],
        # 'cv': [3],
        # 'method': ['sigmoid']
    }
}


## DB_connection

In [18]:
import pyodbc
import pandas as pd

# conection parameters, we understand in real world we would not hardcode them in the code but use environment variables or a secure vault
server = 'DESKTOP-5J0TV08\SQLEXPRESS'  #
database = 'QA'
username = 'herman'  
password = '123456'

# string connection with Windows authentication 
conn_str_windows = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)


# string connection with SQL Server authentication
conn_str_sql = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'UID={username};'
    f'PWD={password};'
)

# we can choose which connection string to use based on our authentication method
conn_str = conn_str_windows

# stablish connection
conn = pyodbc.connect(conn_str)

# we are reading the mortgage credits data to train the model
df_source = pd.read_sql("SELECT * FROM his_credits", conn)
# we are reading the new request of credits to predict the default probability
df_new_requests = pd.read_sql("SELECT * FROM new_requests where DATE = ?", conn, params=[date_param])

df_new_requests.dropna(thresh=13, inplace=True)
df_new_requests_original = df_new_requests.copy()


C:\Users\herma\AppData\Local\Temp\ipykernel_7376\2175735560.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_source = pd.read_sql("SELECT * FROM his_credits", conn)
C:\Users\herma\AppData\Local\Temp\ipykernel_7376\2175735560.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_new_requests = pd.read_sql("SELECT * FROM new_requests where DATE = ?", conn, params=[date_param])


In [19]:
df_new_requests

,DATE,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC
0,2026-02-28,5300.0,75958.0,91703.0,DebtCon,Other,0.0,0.0,0.0,124.182179,1.0,10.0,34.599060
1,2026-02-28,5300.0,97132.0,117343.0,HomeImp,Office,22.0,0.0,0.0,109.933908,1.0,11.0,35.838543
2,2026-02-28,5300.0,47449.0,63895.0,HomeImp,Office,19.0,0.0,0.0,208.162714,0.0,20.0,19.762114
3,2026-02-28,5300.0,50934.0,62957.0,HomeImp,Office,21.0,0.0,0.0,197.241957,0.0,20.0,19.350506
4,2026-02-28,5400.0,86922.0,119846.0,HomeImp,ProfExe,5.0,0.0,0.0,185.076347,1.0,28.0,26.354889
...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,2026-02-28,7000.0,78284.0,98214.0,HomeImp,Other,0.0,0.0,1.0,171.838386,0.0,31.0,36.247963
75,2026-02-28,7000.0,58128.0,93740.0,HomeImp,ProfExe,5.0,0.0,0.0,213.363028,2.0,24.0,34.475302
76,2026-02-28,7000.0,81335.0,81018.0,HomeImp,Other,22.0,0.0,0.0,341.610576,1.0,47.0,27.767300
77,2026-02-28,7000.0,52287.0,87972.0,HomeImp,ProfExe,7.0,0.0,0.0,211.284413,1.0,24.0,33.461402


# Training

## Data Engineering


In [20]:
Thresh_function_drop  =  df_source.notna().sum(axis=1) >= Thresh_parameter  

df_without_drop_values = df_source[Thresh_function_drop].reset_index(drop=True)

df_source_no_drop_values = df_source[~Thresh_function_drop].reset_index(drop=True)

pct_drop_values = len(df_source_no_drop_values) / len(df_source)

# Exercise que use to comfirm that when we have duplicates the alerts is going to pop up
# df_without_drop_values = pd.concat([df_without_drop_values, df_without_drop_values], ignore_index=True)

controls = []

if df_without_drop_values[df_without_drop_values.duplicated()].shape[0] > 0:
    logger.error("There are duplicated rows in the df")
    controls.append({"control": "Duplicated values" , "message": "There are duplicated rows in the df", 'test':len(df_without_drop_values[df_without_drop_values.duplicated()]),'threshold':1 , 'status': 'Failed'})
else:
    controls.append({"control": "Duplicated values" , "message": "No duplicated rows in the df", 'test':len(df_without_drop_values[df_without_drop_values.duplicated()]),'threshold':1 , 'status': 'Passed'})


if pct_drop_values > theshdropvalues:
    logger.error(f"Too many rows dropped: {pct_drop_values:.2%} (threshold: {theshdropvalues:.2%})")
    controls.append({"control": "Drop values threshold" , "message": "Percentage of rows dropped are above threshold" , 'test': pct_drop_values,'threshold': f'{theshdropvalues:.2%}', 'status': 'Failed'})
else:
    controls.append({"control": "Drop values threshold" , "message": "Percentage of rows dropped is acceptable" , 'test': pct_drop_values,'threshold': f'{theshdropvalues:.2%}', 'status': 'Passed'})


    pd.DataFrame(controls)


In [21]:
# we can are going to use 2024 for training and 2025 for testing 

df_without_drop_values['DATE'] =  pd.to_datetime(df_without_drop_values['DATE'], format='%Y%m%d')
X_train = df_without_drop_values[df_without_drop_values['DATE']  < date_param].drop(columns=['DATE','BAD','LOAN'])
y_train = df_without_drop_values[df_without_drop_values['DATE']  < date_param]['BAD']

X_test = df_without_drop_values[df_without_drop_values['DATE']  == date_param].drop(columns=['DATE','BAD','LOAN'])
y_test = df_without_drop_values[df_without_drop_values['DATE']  == date_param]['BAD']
logger.info(f"TOTAL dataset size: {df_without_drop_values.shape[0]} rows, {df_without_drop_values.shape[1]} columns")
logger.info(f"Training set size: {X_train.shape[0]} rows, {X_train.shape[1]} columns")
logger.info(f"Testing  set of {date_param.strftime('%Y-%m-%d')} size: {X_test.shape[0]} rows, {X_test.shape[1]} columns")



# We are ensuring that the columns of the training and testing sets are the same and in the same order
list_columns_train = list(X_train.columns)
X_test = X_test[list_columns_train]
df_new_requests = df_new_requests[list_columns_train]

2026-07-21 21:59:00,499 - INFO - TOTAL dataset size: 5264 rows, 14 columns
2026-07-21 21:59:00,500 - INFO - Training set size: 4533 rows, 11 columns
2026-07-21 21:59:00,503 - INFO - Testing  set of 2026-02-28 size: 255 rows, 11 columns


### column_trans_fit

In [22]:

# We have decided to replace unknown categories values with 'unknown'

X_train['REASON'].fillna('unknown', inplace=True)
X_test['REASON'].fillna('unknown', inplace=True)
X_train['JOB'].fillna('unknown', inplace=True)
X_test['JOB'].fillna('unknown', inplace=True)



category_values = ['REASON', 'JOB']
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),  # fill missing values with the median of each column
    ('rbst_scaler', RobustScaler()),
])


ct = ColumnTransformer(
transformers=[
    ('onehot', OneHotEncoder(sparse_output=True), category_values),  # one-hot encode the 'REASON' and 'JOB' columns
    ('missing',num_pipeline, [  'MORTDUE', 'VALUE', 'YOJ', 'DEROG',
'DELINQ', 'CLAGE', 'NINQ', 'CLNO', 'DEBTINC']),

],
remainder='passthrough'   # by default the columns that are not specified in the transformers list are dropped, but with this parameter we can keep them in the output
,verbose_feature_names_out=False  
)
ct.fit(X_train)
   


ColumnTransformer(remainder='passthrough',
                  transformers=[('onehot', OneHotEncoder(), ['REASON', 'JOB']),
                                ('missing',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('rbst_scaler',
                                                  RobustScaler())]),
                                 ['MORTDUE', 'VALUE', 'YOJ', 'DEROG', 'DELINQ',
                                  'CLAGE', 'NINQ', 'CLNO', 'DEBTINC'])],
                  verbose_feature_names_out=False)

In [23]:
X_train = ct.transform(X_train)
X_train = pd.DataFrame(X_train, columns=ct.get_feature_names_out())
X_test = ct.transform(X_test)
X_test = pd.DataFrame(X_test, columns=ct.get_feature_names_out())


In [24]:

df_new_requests['JOB'].fillna('unknown', inplace=True)
df_new_requests['REASON'].fillna('unknown', inplace=True)


df_new_requests = ct.transform(df_new_requests) 

df_new_requests = pd.DataFrame(df_new_requests, columns=ct.get_feature_names_out())


C:\Users\herma\AppData\Local\Temp\ipykernel_7376\2762322692.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new_requests['JOB'].fillna('unknown', inplace=True)
C:\Users\herma\AppData\Local\Temp\ipykernel_7376\2762322692.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new_requests['REASON'].fillna('unknown', inplace=True)


### column_trans_transform

In [25]:


y_test = y_test.astype(int)
y_train = y_train.astype(int)
X_train[X_train.columns] = X_train[X_train.columns].astype(float)
X_test[X_test.columns] = X_test[X_test.columns].astype(float)

In [26]:

# Example of code if we want to see the correlation between the variables
# df_cor = df_transformed.corr().abs()
# bad_list  = df_cor['BAD'].abs().sort_values(ascending=False)

# bad_list = pd.DataFrame(bad_list.reset_index())
# bad_list.columns = ['Feature', 'Correlation']

# import numpy as np
# for i in category_values:
#    bad_list['Feature'] =  np.where(bad_list['Feature'].str.contains(i), i , bad_list['Feature']) 
   

# bad_list.groupby('Feature')['Correlation'].sum().sort_values(ascending=False)


## model_selection_workflow

### Choosing_top_models

In [27]:



if flag_best_model:
# ============================================
# Evaluar cada modelo
# ============================================

    resultados = []

    for nombre, modelo in dict_model.items():
        print(f"Entrenando {nombre}...")
        
        modelo.fit(X_train, y_train)
        
        # Obtener probabilidades
        if hasattr(modelo, 'predict_proba'):
            probas = modelo.predict_proba(X_test)[:, 1]
        else:
            # Para dict_model que no tienen predict_proba (como SVC sin calibration)
            probas = modelo.decision_function(X_test)
            # Normalizar a [0,1]
            probas = (probas - probas.min()) / (probas.max() - probas.min())
        
        # Métricas para probabilidades
        auc = roc_auc_score(y_test, probas)
        brier = brier_score_loss(y_test, probas)  # <-- Métrica específica para probabilidades
        F1 = f1_score(y_test, modelo.predict(X_test))
        resultados.append({
            'Modelo': nombre,
            'AUC': auc,
            'Brier Score': brier,  # Menor es mejor (0 = perfecto, 0.25 = aleatorio)
            'F1 Score': F1
        })
        
        print(f"  AUC: {auc:.4f} | Brier: {brier:.4f}")

    # ============================================
    # results comparison
    # ============================================

    df_top_5_model = pd.DataFrame(resultados).sort_values('AUC', ascending=False)

    print("\n" + "="*50)
    print("RANKING DE dict_model PARA PREDECIR PROBABILIDADES")
    print("="*50)
    print(df_top_5_model.to_string(index=False))

    # La mejor métrica para probabilidades es BRIER SCORE (no AUC)
    print("\n" + "="*50)
    print("MEJORES dict_model POR BRIER SCORE (menor es mejor)")
    print("="*50)
    print(df_top_5_model.sort_values('Brier Score').to_string(index=False))

### Choosing_best_parametes_model

In [28]:
if flag_best_model:
    df_top_5_model = df_top_5_model.sort_values(['Brier Score','AUC']).reset_index(drop=True).iloc[:5]
    print("="*50)
    print(df_top_5_model.sort_values('F1 Score', ascending=False).to_string(index=False))
    print("="*50)

    top_5_model_performance = []

    for i in range(len(df_top_5_model)):
        model_name = df_top_5_model.iloc[i]['Modelo']
        model_param_grids = param_grids[model_name]
        model =dict_model[model_name]
        grid_search = GridSearchCV(model, model_param_grids, cv=5,
                            scoring='f1_weighted', return_train_score=True)

        grid_search.fit(X_train, y_train)
        parametros = json.dumps((grid_search.best_estimator_.get_params()))
        
        model_adjusted = model.set_params(**grid_search.best_params_).fit(X_train, y_train)
        Y_test_model = model_adjusted.predict(X_test)
        auc = roc_auc_score(y_test, Y_test_model)
        f1 = f1_score(y_test, Y_test_model, average='weighted')
        brier = brier_score_loss(y_test, Y_test_model)  # <-- Métrica específica para probabilidades
        model_results = {
            'model_name': model_name,  'parameters': parametros, 'AUC': auc, 'brier_score': brier, 'f1': f1}
        top_5_model_performance.append(model_results)


In [29]:

# code to create the table in SQL Server:
# CREATE TABLE dbo.model_details (
#     date_analysis DATE,
#     date_run DATETIME,
#     Model_name NVARCHAR(400),
#     parameters NVARCHAR(MAX),
#     AUC FLOAT,
#     Brier_Score FLOAT,
#     f1 FLOAT,
#       model_priority SMALLINT,  
# );


### Loading_model_parameters

In [30]:
if flag_best_model:
    top_5_model_performance  = pd.DataFrame(top_5_model_performance).sort_values('f1', ascending=False)
    print(top_5_model_performance.to_string(index=False))
    top_5_model_performance  = pd.DataFrame(top_5_model_performance).sort_values('f1', ascending=False).reset_index(drop=True)
    top_5_model_performance['model_priority'] = top_5_model_performance.index + 1

    top_5_model_performance['date_analysis'] = date_param
    top_5_model_performance['date_run'] = pd.to_datetime('today')
    top_5_model_performance = top_5_model_performance[[ 'date_analysis', 'date_run','model_name', 'parameters', 'AUC', 'brier_score', 'f1', 'model_priority']]


    data = [tuple(row) for row in top_5_model_performance.to_numpy()]

    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()


    # from datetime import date
    # delte_value = date(2026, 10, 31)

    # cursor.execute("DELETE  FROM model_details WHERE date_analysis = ?", (delte_value))
    # conn.commit()


    logger.info(f"Deleting existing records for date_analysis = {date_param}...")
    cursor.execute("DELETE  FROM model_details WHERE date_analysis = ?", (date_param))
    conn.commit()
    conn.close()

    conn = pyodbc.connect(conn_str)

    cursor = conn.cursor()

    logger.info(f"Inserting {len(data)} rows into model_details...")
    cursor.executemany("""   INSERT INTO model_details (date_analysis, date_run,model_name, parameters, AUC, brier_score, f1,model_priority)
        VALUES (?, ?, ?, ?,?, ?, ?, ?)
    """, data)

    conn.commit()
    conn.close()





## Loading_model

In [31]:

# parametros = json.dumps(grid_search.best_estimator_.get_params())



### Retrieving best model

In [ ]:
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

df_models = pd.read_sql("SELECT * FROM model_details", conn)
df_models['date_analysis'] = pd.to_datetime(df_models['date_analysis'])
# in case if we need to re-run historical data we are ensuring to use the right model used before
df_models = df_models[df_models['date_analysis']<date_param]
df_model = df_models[(df_models['model_priority'] == 1) ].sort_values('date_analysis', ascending=False).reset_index(drop=True).loc[0]


df_model.index = df_model.index.str.lower()
logger.info(f"Best model for date_analysis {date_param}: {df_model['model_name']} with AUC: {df_model['auc']:.4f}, Brier Score: {df_model['brier_score']:.4f}, F1 Score: {df_model['f1']:.4f}")

cursor.close()
df_model

C:\Users\herma\AppData\Local\Temp\ipykernel_7376\2158087203.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_models = pd.read_sql("SELECT * FROM model_details", conn)


0     True
1     True
2     True
3     True
4     True
5    False
6    False
7    False
8    False
9    False
Name: date_analysis, dtype: bool

In [731]:
# Control to check if the model is still valid for deployment (if the date of analysis is less than 6 months old)

if date_param - pd.DateOffset(months=6) <= df_model['date_analysis']:
    logger.info(f"Model {df_model['model_name']} is still valid for deployment (analyzed on  {df_model['date_analysis'].strftime('%Y-%m-%d')} on date  {date_param.strftime('%Y-%m-%d')})")
else:
    logger.warning(f"Model {df_model['model_name']} is outdated for deployment (analyzed on {df_model['date_analysis'].strftime('%Y-%m-%d')})")    




2026-07-15 20:54:37,139 - INFO - Model LightGBM is still valid for deployment (analyzed on  2026-03-31 on date  2026-05-31)


In [732]:
logger.info(f"Loading best model parameters")
model_parameters = json.loads(df_model['parameters'])

logger.info(f"Setting parameters for {df_model['model_name']}")

# model_parameters = {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 5, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 100, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}
act_model = dict_model[df_model['model_name']].set_params(**model_parameters).fit(X_train, y_train)

2026-07-15 20:54:37,189 - INFO - Loading best model parameters
2026-07-15 20:54:37,191 - INFO - Setting parameters for LightGBM


[LightGBM] [Info] Number of positive: 1061, number of negative: 4098
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000690 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1199
[LightGBM] [Info] Number of data points in the train set: 5159, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.205660 -> initscore=-1.351287
[LightGBM] [Info] Start training from score -1.351287
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [733]:
model_parameters

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': 5,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'num_leaves': 31,
 'objective': None,
 'random_state': 42,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0}

### Evaluating_new_historical_data

In [734]:
Y_act_model = act_model.predict(X_test)

auc = roc_auc_score(y_test, Y_act_model)
brier = brier_score_loss(y_test, Y_act_model)  # <-- Métrica específica para probabilidades
F1 = f1_score(y_test, Y_act_model)

logger.info(f"Best model performance on test set: AUC: {auc:.4f}, Brier Score: {brier:.4f}, F1 Score: {F1:.4f}")


df_model_performance = df_model.copy()
df_model_performance =pd.DataFrame(df_model_performance.transpose()).transpose()
df_model_performance['date_run'] = pd.to_datetime(pd.to_datetime('today'))
df_model_performance['date_analysis']= date_param
df_model_performance['auc'] = auc
df_model_performance['brier_score'] = brier
df_model_performance['f1'] = F1
df_model_performance= df_model_performance[['date_analysis', 'date_run', 'model_name', 'parameters', 'auc',
       'brier_score', 'f1']]


data = [tuple(row) for row in df_model_performance.to_numpy()]
# data = tuple(df_model_performance)
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

cursor.execute("DELETE  FROM current_model_performance WHERE date_analysis = ?", (date_param))
conn.commit()



cursor.executemany("""   INSERT INTO current_model_performance (date_analysis,date_run, model_name, parameters, auc,
       brier_score, f1   )
    VALUES (?, ?, ?, ?,?, ?, ?)
""", data)

conn.commit()





# df_model_performance.drop(columns ={'model_priority'}, inplace=True)
# df_model_performance.columns

2026-07-15 20:54:37,702 - INFO - Best model performance on test set: AUC: 0.7275, Brier Score: 0.1429, F1 Score: 0.6154


In [735]:
df_model_performance['date_analysis']

0   2026-05-31
Name: date_analysis, dtype: datetime64[ns]

In [736]:
# code to create the table in SQL Server that saves the performances of the actual model using the new historical data. 
# CREATE TABLE dbo.current_model_performance (
#     date_analysis DATE,
#     date_run DATETIME,
#     Model_name NVARCHAR(400),
#     parameters NVARCHAR(MAX),
#     AUC FLOAT,
#     Brier_Score FLOAT,
#     f1 FLOAT,
#    
# );


### Predecting_new_requests

In [737]:
Y_new_requests = act_model.predict_proba(df_new_requests).round(2)
Y_new_requests_ = act_model.predict(df_new_requests).round(2)
Y_new_requests
pd.set_option('display.max_rows', None)
y_probability  = pd.concat([pd.DataFrame(Y_new_requests),pd.DataFrame(Y_new_requests_)], axis=1)  
y_probability.columns = ['Prob_No_Default', 'Prob_Default', 'Flag_Default']
# y_probability.sort_values('Prob_Default', ascending=False)


df_new_requestst_upload = pd.concat([df_new_requests_original.reset_index(drop=True), y_probability], axis=1)


In [738]:
df_new_requestst_upload['date_execution'] = pd.Timestamp.now()

In [739]:
df_new_requestst_upload['date_run'] = pd.to_datetime(df_new_requestst_upload['date_execution']).dt.strftime('%Y-%m-%d %H:%M:%S')

In [740]:
df_new_requestst_upload = df_new_requestst_upload[['DATE', 'date_execution', 'LOAN', 'MORTDUE', 'VALUE', 'REASON', 'JOB', 'YOJ',
       'DEROG', 'DELINQ', 'CLAGE', 'NINQ', 'CLNO', 'DEBTINC',
       'Prob_No_Default', 'Prob_Default', 'Flag_Default' ]]

In [741]:

#  CREATE TABLE dbo.new_requests_results (
#      date_analysis DATE,
#    date_run DATETIME,
#     LOAN bigint,
#     MORTDUE FLOAT,
#    VALUE FLOAT,
#    REASON varchar(400),
#     JOB varchar(400),
# YOJ FLOAT,
# DEROG FLOAT,
# DELINQ FLOAT,
# CLAGE FLOAT,
# NINQ FLOAT,
# CLNO FLOAT,
# DEBTINC FLOAT,
#   Prob_No_Default FLOAT,
#   Prob_Default FLOAT,
#   Flag_Default int,
#  );



#  CREATE TABLE dbo.new_requests_results (
#      date_analysis DATE,
#    date_run DATETIME,
#     LOAN bigint,
#     MORTDUE FLOAT,
#    VALUE FLOAT,
#    REASON varchar(400),
#     JOB varchar(400),
# YOJ FLOAT,
# DEROG FLOAT,
# DELINQ FLOAT,
# CLAGE FLOAT,
# NINQ FLOAT,
# CLNO FLOAT,
# DEBTINC FLOAT,
#   Prob_No_Default FLOAT,
#   Prob_Default FLOAT,
#   Flag_Default int,
#  );

In [742]:
df_new_requestst_upload.head()
df_new_requestst_upload.rename(columns={'DATE': 'date_analysis'}, inplace=True)


In [743]:
# for row in df_new_requestst_upload.to_numpy():
#     print(tuple(row))

In [744]:

data = [tuple(row) for row in df_new_requestst_upload.to_numpy()]

cursor = conn.cursor()


# from datetime import date
# delte_value = date(2026, 10, 31)

# cursor.execute("DELETE  FROM model_details WHERE date_analysis = ?", (delte_value))
# conn.commit()


# logger.info(f"Deleting existing records for date_analysis = {date_param}...")
cursor.execute("DELETE  FROM new_requests_results WHERE date_analysis = ?", (date_param))
conn.commit()

# logger.info(f"Inserting {len(data)} rows into new_requests_results")
cursor.executemany("""   INSERT INTO new_requests_results (date_analysis, date_run,
                   LOAN, MORTDUE, VALUE, REASON, JOB, YOJ,
                   DEROG, DELINQ, CLAGE, NINQ, CLNO, DEBTINC, Prob_No_Default, Prob_Default, Flag_Default
               )
    VALUES (?, ?, ?, ?,?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", data)

conn.commit()




cursor.close()
conn.close()

In [745]:
df_new_requestst_upload


,date_analysis,date_execution,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,Prob_No_Default,Prob_Default,Flag_Default
0,2026-05-31,2026-07-15 20:54:37.994507,8900.0,64343.0,84013.0,DebtCon,Other,4.0,0.0,0.0,39.221719,0.0,12.0,35.788437,0.91,0.09,0
1,2026-05-31,2026-07-15 20:54:37.994507,8900.0,133847.0,157399.0,HomeImp,ProfExe,17.0,0.0,2.0,70.884144,0.0,12.0,34.702331,0.44,0.56,1
2,2026-05-31,2026-07-15 20:54:37.994507,8900.0,147801.0,157335.0,HomeImp,Other,4.0,0.0,0.0,85.574866,4.0,21.0,34.756560,0.77,0.23,0
3,2026-05-31,2026-07-15 20:54:37.994507,8900.0,77261.0,96434.0,DebtCon,ProfExe,3.0,0.0,0.0,144.037004,0.0,26.0,33.908466,0.97,0.03,0
4,2026-05-31,2026-07-15 20:54:37.994507,9000.0,50513.0,61190.0,DebtCon,Other,5.0,0.0,0.0,75.388551,1.0,11.0,30.498005,0.94,0.06,0
5,2026-05-31,2026-07-15 20:54:37.994507,9000.0,75453.0,100646.0,DebtCon,ProfExe,3.0,0.0,0.0,126.309835,0.0,26.0,33.758466,0.97,0.03,0
6,2026-05-31,2026-07-15 20:54:37.994507,9000.0,81248.0,92503.0,DebtCon,Other,0.0,0.0,0.0,121.199650,2.0,10.0,37.390240,0.96,0.04,0
7,2026-05-31,2026-07-15 20:54:37.994507,9000.0,90969.0,108606.0,HomeImp,Mgr,5.0,5.0,2.0,113.257509,5.0,28.0,27.347597,0.06,0.94,1
8,2026-05-31,2026-07-15 20:54:37.994507,9000.0,79259.0,84322.0,HomeImp,Other,22.0,0.0,0.0,359.332272,1.0,47.0,29.671390,0.99,0.01,0
9,2026-05-31,2026-07-15 20:54:37.994507,9000.0,56162.0,75526.0,HomeImp,Other,6.0,0.0,0.0,127.786970,0.0,15.0,35.734057,0.93,0.07,0
